# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JaudatUllahKhan/my-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

In [13]:
# Setup and Task Type Definition
import pandas as pd
import numpy as np

task_type = "Binary Classification & Scoring"
lane = "Lane 1 - Content Refresh & Decay"

print(f"Lane: {lane}")
print(f"ML Task Type: {task_type}")

Lane: Lane 1 - Content Refresh & Decay
ML Task Type: Binary Classification & Scoring


**Task Type: Binary Classification & Scoring / Ranking**

For **Lane 1 (Content Refresh & Decay)**, the core ML task is framed as a combination of **Binary Classification** and **Scoring/Ranking**:

1. **Classification:** Predict whether a page is experiencing meaningful performance decay (`1` = Declining, `0` = Stable/Growing).
2. **Scoring / Ranking:** Output predicted decay probabilities ($\hat{y} \in [0, 1]$) to rank pages by decay severity, allowing content teams to prioritize high-risk, high-exposure pages first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

In [14]:
# Target definition check
target_col = "is_declining_label"
print(f"Target column defined as: {target_col}")

Target column defined as: is_declining_label


* **Target Label (`is_declining_label`):** A binary indicator derived from the dataset (`1` if `trend_direction == 'down'`, `0` otherwise).
* **Proxy Nature:** We do not directly observe "content staleness" or "loss of user intent alignment." Instead, we use a 90-day trajectory in Google Search Console performance signals (impressions, clicks, CTR, position) as an observable proxy label for page decay.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

In [15]:
# Metric definition check
primary_metric = "Precision@K (K=20, K=50)"
target_good_threshold = 0.80

print(f"Primary Metric: {primary_metric}")
print(f"Operational 'Good' Target: {target_good_threshold:.0%}")

Primary Metric: Precision@K (K=20, K=50)
Operational 'Good' Target: 80%


* **Primary Metric: Precision@K (specifically Precision@20 and Precision@50)**
  * **Why:** Content teams have limited editorial bandwidth to rewrite pages each month. Precision@K measures the fraction of true declining pages among the top $K$ pages recommended by the model. Maximizing Precision@K avoids wasting editor time on healthy pages. A Precision@20 $> 0.80$ (80%+ true positives in the top 20 recommendations) represents a "good" operational target.
* **Secondary Metric: ROC-AUC / PR-AUC**
  * Evaluates overall ranking quality across varying decision thresholds without relying on a single cutoff point.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [16]:
import os, sys, subprocess
import pandas as pd
import numpy as np

# Ensure working directory is set correctly in Colab
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("../..")

# Load starter dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Create binary target label
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)

# Detect actual column names present in the dataset
id_col = "page_id" if "page_id" in df.columns else df.columns[0]
candidate_cols = [id_col, "content_age_days", "days_since_last_update",
                  "impressions_90d", "ctr", "avg_position", "is_declining_label"]
display_cols = [col for col in candidate_cols if col in df.columns]

print(f"Total Rows (Unit of Analysis = 1 Page): {len(df):,}")
df[display_cols].head(5)

Total Rows (Unit of Analysis = 1 Page): 30,000


,content_id,content_age_days,days_since_last_update,impressions_90d,ctr,avg_position,is_declining_label
0,content_304f48230142,187,20,3803,0.76,10.6,1
1,content_a1fb4e703a9e,445,25,15320,0.05,20.3,1
2,content_9aa793d4d895,141,20,12581,0.09,36.5,1
3,content_331d6c4de07b,463,22,11751,0.49,6.2,0
4,content_d99b7a2d90ca,263,14,19140,0.13,44.0,1


**Unit of Analysis:** One row = **One unique web page (`page_hash_id`)**.

Below, we load the starter dataset and construct our target label (`is_declining_label`) explicitly to display the exact shape and features of our unit of analysis dataframe.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

In [17]:
# Validation check comparing a naive hard-coded rule against true dataset labels
naive_rule = (df['days_since_last_update'] > 180) & (df['impressions_90d'] > 1000)
rule_precision = (df[naive_rule]['is_declining_label'] == 1).mean()

print(f"Naive Rule Precision: {rule_precision:.1%}")
print("Shows why non-linear features and calibrated scores are required beyond simple IF-statements.")

Naive Rule Precision: 91.7%
Shows why non-linear features and calibrated scores are required beyond simple IF-statements.


1. **Non-linear Multi-Feature Interactions:** Simple heuristics like `days_since_last_update > 180` fail because a high-exposure page losing CTR after 60 days is far more urgent than a 2-year-old page with zero traffic.
2. **Adaptive Weighting Across Features:** Machine learning models balance multiple continuous signals simultaneously (impressions, ranking drops, content age, CTR) rather than relying on arbitrary hard cutoffs.
3. **Calibrated Probability Scores:** ML provides continuous output scores ($\hat{y} \in [0, 1]$), allowing content teams to dynamically adjust how many pages they review based on weekly editorial capacity.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.